### Setup

In [1]:
from pathlib import Path
import sys
import numpy as np


def find_project_root():
    current = Path.cwd().resolve()

    for candidate in [
        current,
        *current.parents
    ]:
        if (
            (candidate / "main.py").exists()
            and
            (candidate / "utils").exists()
            and
            (candidate / "thesis").exists()
        ):
            return candidate

    raise RuntimeError(
        "PyCIL project root not found."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )

from utils.data_manager import DataManager

print(PROJECT_ROOT)

C:\Skripsi\Project\PyCIL


### Test AID T=5

In [2]:
aid_dm = DataManager(
    dataset_name="aid",
    shuffle=False,
    seed=1993,
    init_cls=14,
    increment=4,
    dataset_root=(
        PROJECT_ROOT
        / "datasets/raw/AID"
    ),
    split_root=(
        PROJECT_ROOT
        / "splits/aid/seed_1993"
    ),
    task_config_path=(
        PROJECT_ROOT
        / "splits/aid/seed_1993/T5_tasks.json"
    ),
)

In [3]:
print("Tasks:", aid_dm.nb_tasks)

print(
    "Task sizes:",
    [
        aid_dm.get_task_size(i)
        for i in range(
            aid_dm.nb_tasks
        )
    ]
)

print(
    "Class order:",
    aid_dm._class_order
)

Tasks: 5
Task sizes: [14, 4, 4, 4, 4]
Class order: [27, 21, 26, 2, 8, 13, 10, 6, 15, 20, 14, 23, 12, 28, 11, 5, 0, 22, 18, 4, 19, 24, 9, 7, 16, 3, 29, 17, 25, 1]


### Test Task 0 AID

In [4]:
aid_task0 = aid_dm.get_dataset(
    indices=np.arange(
        aid_dm.get_task_size(0)
    ),
    source="train",
    mode="train",
)

print(
    "Task 0 train images:",
    len(aid_task0)
)

print(
    "Unique incremental labels:",
    np.unique(
        aid_task0.labels
    )
)

print(
    "Image path example:",
    aid_task0.images[0]
)

Task 0 train images: 3656
Unique incremental labels: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13]
Image path example: C:\Skripsi\Project\PyCIL\datasets\raw\AID\Stadium\stadium_1.jpg


In [5]:
index, image, label = aid_task0[0]

print("Index :", index)
print("Shape :", image.shape)
print("Label :", label)

Index : 0
Shape : torch.Size([3, 600, 600])
Label : 0


### Incremental label mapping test

In [6]:
first_original_class = (
    aid_dm._class_order[0]
)

print(
    "First original class:",
    first_original_class
)

mapped_targets = (
    aid_dm._train_targets
)

print(
    "Mapped target range:",
    mapped_targets.min(),
    "-",
    mapped_targets.max()
)

assert set(
    np.unique(mapped_targets)
) == set(range(30))

print(
    "Incremental label mapping: PASSED"
)

First original class: 27
Mapped target range: 0 - 29
Incremental label mapping: PASSED


### NWPU Test

In [7]:
nwpu_dm = DataManager(
    dataset_name="nwpu",
    shuffle=False,
    seed=1993,
    init_cls=25,
    increment=5,
    dataset_root=(
        PROJECT_ROOT
        / "datasets/raw/NWPU-RESISC45"
    ),
    split_root=(
        PROJECT_ROOT
        / "splits/nwpu/seed_1993"
    ),
    task_config_path=(
        PROJECT_ROOT
        / "splits/nwpu/seed_1993/T5_tasks.json"
    ),
)

print(
    "NWPU tasks:",
    nwpu_dm.nb_tasks
)

print(
    "Task sizes:",
    [
        nwpu_dm.get_task_size(i)
        for i in range(
            nwpu_dm.nb_tasks
        )
    ]
)

NWPU tasks: 5
Task sizes: [25, 5, 5, 5, 5]


#### Task 0

In [8]:
nwpu_task0 = nwpu_dm.get_dataset(
    indices=np.arange(25),
    source="train",
    mode="train",
)

print(
    "Task 0 images:",
    len(nwpu_task0)
)

_, image, label = nwpu_task0[0]

print(
    "Image shape:",
    image.shape
)

Task 0 images: 14000
Image shape: torch.Size([3, 256, 256])


### Test centralized loaders

In [ ]:
from thesis.data.loaders import (
    build_train_loader,
    build_eval_loader,
)


aid_loader = build_train_loader(
    dataset=aid_task0,
    dataset_name="aid",
    batch_size=32,
    num_workers=0,
    seed=1993,
)

print(
    type(aid_loader.sampler)
)

In [ ]:
nwpu_loader = build_train_loader(
    dataset=nwpu_task0,
    dataset_name="nwpu",
    batch_size=32,
    num_workers=0,
    seed=1993,
)

print(
    type(nwpu_loader.sampler)
)

In [ ]:
aid_eval_dataset = aid_dm.get_dataset(
    indices=np.arange(14),
    source="test",
    mode="test",
)

aid_eval_loader = build_eval_loader(
    dataset=aid_eval_dataset,
    batch_size=32,
    num_workers=0,
)

print(
    type(aid_eval_loader.sampler)
)